# MNISym_coreg_regression

**Pipeline**

Using full-image coregistered anatomicals with their segementations normalized to MNI Symmetric template:

- run regression on each of: wm, gm, T1, csf

- for subjects with left hemisphere lesion, flip (along x-axis, L-R flip) their slope image
    
    - slope image for each of wm, gm, T1, csf

- get the average image for patients and controls

In [1]:
# Imports

# need path to root directory
import sys
sys.path.append('/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/')

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl

import nitools as nt
from smarts_cerebellum import regression
from image_processing import overall_image
from image_processing import mirror_lesion
from pipelines import MNISym_coreg_regression
from image_processing import tissue_extractor as te
import smarts_cerebellum.globals as gl

from pathlib import Path
import os

In [2]:
# directories
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

## Make a summarized dataframe for ROIs
Where ROIs are defined in the SUIT

In [11]:
search_path = os.path.join(gl.baseDir, 'Regression')

# files to include in summarized dataframe


In [ ]:
# use flipped images for those with lesions on the left side

In [4]:
# mirror images
from image_processing import mirror_lesion

In [7]:
mirror_lesion.FlipLR?

Signature: mirror_lesion.FlipLR(image)
Docstring:
Simple flip: flips image along x-axis (L-R flip)

Input: image (Nifti or string)

Output: Nifti image
File:      ~/Documents/GitHub/smarts_cerebellum/image_processing/mirror_lesion.py
Type:      function

In [ ]:
suffixes = [
    'MNISym_CSF_coreg_reslice_slope.nii.gz',
    'MNISym_WM_coreg_reslice_slope.nii.gz',
    'MNISym_GM_coreg_reslice_slope.nii.gz',
    'MNISym_T1_coreg_reslice_slope.nii.gz'
]

In [8]:
left_lesion_df = p_df[p_df.LesionSide == 'left ']

In [12]:
search_path

'/cifs/diedrichsen/data/smarts_cerebellum/Regression'

In [16]:
for subj in left_lesion_df.subj_id.unique():
    flip_csf = f'{search_path}/{subj}/{subj}_MNISym_CSF_coreg_reslice_slope.nii.gz'
    if not Path(flip_csf).is_file():
        print(f'Skip {subj}')
        continue
    flipped_csf = mirror_lesion.FlipLR(flip_csf)
    nib.save(flipped_csf, f'{search_path}/{subj}/{subj}_MNISym_CSF_coreg_reslice_slope_FlipLR.nii.gz')


Skip CU_2697
Skip JHU_2374


In [17]:
type = 'WM'
for subj in left_lesion_df.subj_id.unique():
    flip = f'{search_path}/{subj}/{subj}_MNISym_{type}_coreg_reslice_slope.nii.gz'
    if not Path(flip).is_file():
        print(f'Skip {subj}')
        continue
    flipped = mirror_lesion.FlipLR(flip)
    flipped = f'{search_path}/{subj}/{subj}_MNISym_{type}_coreg_reslice_slope_FlipLR.nii.gz'

Skip CU_2697
Skip JHU_2374


In [19]:
type = 'T1'
for subj in left_lesion_df.subj_id.unique():
    flip = f'{search_path}/{subj}/{subj}_MNISym_{type}_coreg_reslice_slope.nii.gz'
    if not Path(flip).is_file():
        print(f'Skip {subj}')
        continue
    flipped = mirror_lesion.FlipLR(flip)
    flipped = f'{search_path}/{subj}/{subj}_MNISym_{type}_coreg_reslice_slope_FlipLR.nii.gz'

Skip CU_2697
Skip JHU_2374


In [18]:
type = 'GM'
for subj in left_lesion_df.subj_id.unique():
    flip = f'{search_path}/{subj}/{subj}_MNISym_{type}_coreg_reslice_slope.nii.gz'
    if not Path(flip).is_file():
        print(f'Skip {subj}')
        continue
    flipped = mirror_lesion.FlipLR(flip)
    flipped = f'{search_path}/{subj}/{subj}_MNISym_{type}_coreg_reslice_slope_FlipLR.nii.gz'

Skip CU_2697
Skip JHU_2374


In [21]:
from image_processing import utils
import SUITPy as suit


suffixes = [
    'MNISym_CSF_coreg_reslice_slope.nii.gz',
    'MNISym_WM_coreg_reslice_slope.nii.gz',
    'MNISym_GM_coreg_reslice_slope.nii.gz',
    'MNISym_T1_coreg_reslice_slope.nii.gz'
]
flipped_suffixes = [
    'MNISym_CSF_coreg_reslice_slope_FlipLR.nii.gz',
    'MNISym_WM_coreg_reslice_slope_FlipLR.nii.gz',
    'MNISym_GM_coreg_reslice_slope_FlipLR.nii.gz',
    'MNISym_T1_coreg_reslice_slope_FlipLR.nii.gz'
]

def descriptive_dataframe(atlas_df, p_df, subj):
    """
    Creates a descriptive dataframe
    UNIQUE SUBJECT IMAGE, NOT WEEKS! --> weeks under construction!

    Inputs:
        atlas_df (Pandas dataframe): dataframe from atlas summary
        p_info (Pandas dataframe): dataframe with descriptive information for participants

    Outputs:
        atlas_df (Pandas dataframe): updated dataframe (with descriptive information)
    """

    # try doing this for only one subject; then, we can loop through in the call
    refT1 = p_df[p_df.subj_id == subj]['RefT1'].iloc[0].strip()
    subj_df = p_df[(p_df.subj_id == subj) & (p_df.Week.str.strip() == refT1)]


    # mask the row to which we are adding data
    row_mask = (atlas_df.subj_id == subj)
    print(f'Writing data for {subj}')

    atlas_df.loc[row_mask, 'ID'] = subj_df['ID'].values[0]
    atlas_df.loc[row_mask, 'Centre'] = subj_df['Centre'].values[0]
    atlas_df.loc[row_mask, 'RefT1'] = refT1
    atlas_df.loc[row_mask, 'age'] = subj_df['age'].values[0]
    atlas_df.loc[row_mask, 'Gender'] = subj_df.Gender.values[0]
    atlas_df.loc[row_mask, 'isPatient'] = subj_df.isPatient.values[0]
    atlas_df.loc[row_mask, 'LesionSide'] = subj_df.LesionSide.values[0]
    atlas_df.loc[row_mask, 'LesionLocation'] = subj_df.LesionLocation.values[0]
    atlas_df.loc[row_mask, 'handedness'] = subj_df.handedness.values[0]

 
    return atlas_df


# Make summarized dataframe
def make_summarized_dataframe(p_df,
                              search_path,
                              the_atlas, maps, space,
                              ):
    """
    Make full summarized dataframe that has: ROIs for each subject, along with descriptive information

    Inputs:
        p_df (Pandas dataframe): info file for participants
        search_path (str): directory where files are stored (parent directory for all subjects)
        suffixes (tuple of str): suffixes for all files you want to find

        the_atlas (str): cerebellar atlas --> see SUITPy
        maps (str): map to use in summarizing (cerebellar map) --> see SUITPy
        space: space of the files


    Outputs:

    """

    dfs = []

    # loop through all subjects - perform each operation on each subject
    for subj in p_df.subj_id.unique():
        # find their files - returns string list of files
        if subj in left_lesion_df.subj_id.unique():
            file_list = utils.file_search(search_path = search_path, subj_id = subj, suffixes = flipped_suffixes)
        else:
            file_list = utils.file_search(search_path = search_path, subj_id = subj, suffixes = suffixes)


        
        if not file_list:
            continue # skip subjects without the files
        
        # summarize volume in each ROI for each file type
        
        df = suit.summarize_data(
                                 images = file_list,
                                 atlas = the_atlas,
                                 maps = maps,
                                 space = space,
                                 stats = ['mean', 'nansum']

        )
        
        df['subj_id']= subj

        # then make the descriptive dataframe for each subject
        descriptive_df = descriptive_dataframe(atlas_df = df, p_df = p_df, subj = subj)

        # add all dataframes to the list
        dfs.append(descriptive_df)

    # combine all of them
    all_df = pd.concat(dfs, ignore_index = True)

    

    return all_df

In [23]:
summarized_df = make_summarized_dataframe(p_df = p_df,
                                             search_path = search_path,
                                
                                             the_atlas = 'Diedrichsen_2009',
                                             maps = 'atl-Anatom',
                                             space = 'MNISym'
                                             )


Writing data for CU_2310
Writing data for CU_2538
Writing data for CU_2663
Writing data for CU_2925
Writing data for JHU_2282
Writing data for JHU_2395
Writing data for JHU_2531
Writing data for JHU_2577
Writing data for JHU_2650
Writing data for JHU_2684
Writing data for JHU_2713
Writing data for JHU_2789
Writing data for JHU_3175
Writing data for JHU_3176
Writing data for UZ_2365
Writing data for UZ_2450
Writing data for UZ_2565
Writing data for UZ_2595
Writing data for UZ_2652
Writing data for UZ_2654
Writing data for UZ_2906
Writing data for UZ_3030
Writing data for UZ_3057
Writing data for UZ_3151
Writing data for UZ_3158
Writing data for UZ_3166
Writing data for UZ_3224
Writing data for UZ_3226
Writing data for UZ_3227
Writing data for UZ_3238
Writing data for UZ_3239
Writing data for UZ_3240
Writing data for UZ_3241
Writing data for UZ_3243
Writing data for UZ_3246
Writing data for UZ_3247
Writing data for UZ_3248
Writing data for CUP_1001
Writing data for CUP_1002
Writing data 

In [24]:
# save the dataframe

summarized_df.to_csv(f'{search_path}/MNISym_coreg_slope_flipLesion_AtlasSUIT_summarized.tsv', mode = 'w', sep = '\t', index = False, header = True)

In [ ]:
summarized_df

,image,image_name,frame,region,regionname,volume,atlas,map,space,mean,...,subj_id,ID,Centre,RefT1,age,Gender,isPatient,LesionSide,LesionLocation,handedness
0,1,CU_2310_MNISym_CSF_coreg_reslice_slope.nii.gz,0,1,Left_I_IV,4567.0,Diedrichsen_2009,atl-Anatom,MNISym,-0.000177,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
1,1,CU_2310_MNISym_CSF_coreg_reslice_slope.nii.gz,0,2,Right_I_IV,5524.0,Diedrichsen_2009,atl-Anatom,MNISym,0.000065,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
2,1,CU_2310_MNISym_CSF_coreg_reslice_slope.nii.gz,0,3,Left_V,5854.0,Diedrichsen_2009,atl-Anatom,MNISym,-0.000009,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
3,1,CU_2310_MNISym_CSF_coreg_reslice_slope.nii.gz,0,4,Right_V,5975.0,Diedrichsen_2009,atl-Anatom,MNISym,0.000108,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
4,1,CU_2310_MNISym_CSF_coreg_reslice_slope.nii.gz,0,5,Left_VI,12898.0,Diedrichsen_2009,atl-Anatom,MNISym,-0.000046,...,CU_2310,2310.0,CU,W0,57.0,M,1.0,left,subcortical,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6659,4,UZP_1008_MNISym_T1_coreg_reslice_slope.nii.gz,0,30,Right_Dentate,2197.0,Diedrichsen_2009,atl-Anatom,MNISym,-1.731406,...,UZP_1008,1008.0,UZP,W0,63.0,M,0.0,none,none,2.0
6660,4,UZP_1008_MNISym_T1_coreg_reslice_slope.nii.gz,0,31,Left_Interposed,255.0,Diedrichsen_2009,atl-Anatom,MNISym,-1.445008,...,UZP_1008,1008.0,UZP,W0,63.0,M,0.0,none,none,2.0
6661,4,UZP_1008_MNISym_T1_coreg_reslice_slope.nii.gz,0,32,Right_Interposed,277.0,Diedrichsen_2009,atl-Anatom,MNISym,-1.595848,...,UZP_1008,1008.0,UZP,W0,63.0,M,0.0,none,none,2.0
6662,4,UZP_1008_MNISym_T1_coreg_reslice_slope.nii.gz,0,33,Left_Fastigial,2.0,Diedrichsen_2009,atl-Anatom,MNISym,-0.639360,...,UZP_1008,1008.0,UZP,W0,63.0,M,0.0,none,none,2.0


In [11]:
# save the dataframe

summarized_df.to_csv(f'{search_path}/MNISym_coreg_slope_AtlasSUIT_summarized.tsv', mode = 'w', sep = '\t', index = False, header = True)

In [10]:
search_path

'/cifs/diedrichsen/data/smarts_cerebellum/Regression'

In [7]:
len(summarized_df.image_name.unique())

196

In [9]:
49*4

196